In [1]:
import pandas as pd

In [2]:
ver = 'Jan_4_24'
df = pd.read_csv(f'/data/pgangwar/vaccinescout/data/{ver}/mutations.tsv', sep = '\t')

In [3]:
df = df[['mutation', 'R / R_A']]

In [5]:
df = df.dropna(subset=['mutation'])
df = df[df['mutation'].str.startswith('S:')]


In [6]:
print(len(df))

1496


In [8]:
import re
from collections import defaultdict
import pandas as pd
import numpy as np

def expand_mutations(df):
    """
    Expand df (with columns 'mutation' and 'R / R_A') by:
      - adding inverses: S:XnY -> S:YnX with score = 1/score(XnY)
      - adding chains within same site: if X->Y and Y->Z exist, add X->Z with score = score(X->Y)*score(Y->Z)
    Strictly disallows self transitions (X->X) and never overwrites existing records.
    """

    # --- Step 1. Drop NaN and ensure numeric scores ---
    df = df.dropna(subset=['mutation', 'R / R_A']).copy()
    df['R / R_A'] = pd.to_numeric(df['R / R_A'], errors='coerce')
    df = df.dropna(subset=['R / R_A']).copy()

    # --- Step 2. Parse mutation strings (no spaces expected) ---
    pat = re.compile(r'^S:([A-Za-z])(\d+)([A-Za-z])$')
    parsed_rows = []
    for _, row in df.iterrows():
        m = pat.match(row['mutation'])
        if not m:
            continue  # skip invalid mutation strings
        old_aa = m.group(1).upper()
        site = int(m.group(2))
        new_aa = m.group(3).upper()
        score = float(row['R / R_A'])
        parsed_rows.append((site, old_aa, new_aa, score))

    # --- Step 3. Build base mapping ---
    orig_edges = {(site, old, new): score for site, old, new, score in parsed_rows}
    edges = dict(orig_edges)  # working copy

    # --- Step 4. Add inverses ---
    for (site, a, b), sc in list(orig_edges.items()):
        if a == b or sc == 0:  # skip self or zero-score cases
            continue
        inv_key = (site, b, a)
        if inv_key not in orig_edges and inv_key not in edges:
            edges[inv_key] = 1.0 / sc

    # --- Step 5. Transitive closure within each site ---
    made_progress = True
    iter_count = 0
    while made_progress:
        made_progress = False
        iter_count += 1

        # index edges by site
        edges_by_site = defaultdict(list)
        for (site, x, y), sc in edges.items():
            edges_by_site[site].append((x, y, sc))

        # per site, add inferred transitions
        for site, edge_list in edges_by_site.items():
            outgoing = defaultdict(list)
            for x, y, sc in edge_list:
                outgoing[x].append((y, sc))

            for x, y, sc_xy in edge_list:
                for z, sc_yz in outgoing.get(y, []):
                    if z == x:  # no self transitions
                        continue
                    new_key = (site, x, z)
                    if new_key in edges or new_key in orig_edges:
                        continue
                    new_score = sc_xy * sc_yz
                    if not np.isfinite(new_score):
                        continue
                    edges[new_key] = float(new_score)
                    made_progress = True

    # --- Step 6. Construct final DataFrame ---
    rows = []
    for (site, a, b), sc in edges.items():
        if a == b:  # enforce no self transitions
            continue
        mutation_str = f"S:{a}{site}{b}"
        sc_final = orig_edges.get((site, a, b), sc)
        rows.append({'mutation': mutation_str, 'R / R_A': sc_final})

    df_out = pd.DataFrame(rows).drop_duplicates(subset=['mutation']).reset_index(drop=True)
    return df_out


In [9]:
df_ = expand_mutations(df)

In [10]:
dupes = df_[df_.duplicated(subset=['mutation'], keep=False)]
if not dupes.empty:
    print("Warning: duplicate mutations found!")
    print(dupes)


In [11]:
self_transitions = df_[df_['mutation'].str[2] == df_['mutation'].str[-1]]
if not self_transitions.empty:
    print("Warning: self transitions detected!")
    print(self_transitions)


In [12]:
import re
pattern = re.compile(r'^S:[A-Z]\d+[A-Z]$')
invalid = df_[~df_['mutation'].str.match(pattern)]
if not invalid.empty:
    print("Warning: invalid mutation strings found!")
    print(invalid)


In [20]:
if df_['R / R_A'].isna().any():
    print("Warning: NaN scores detected")
if not np.issubdtype(df_['R / R_A'].dtype, np.number):
    print("Warning: scores are not numeric")


In [16]:
df_['site'] = df_['mutation'].str[3:-1].astype(int)
summary = df_.groupby('site').agg(
    num_mutations=('mutation', 'count'),
    min_score=('R / R_A', 'min'),
    max_score=('R / R_A', 'max')
)
print(summary)
df_.drop(columns=['site'], inplace=True)  # clean up


      num_mutations  min_score  max_score
site                                     
3                 2   0.999121   1.000880
6                 6   0.999951   1.000049
8                 6   0.999970   1.000030
9                 6   0.976477   1.024090
12                2   1.000000   1.000000
...             ...        ...        ...
1263             12   0.996085   1.003930
1264              2   0.976305   1.024270
1265              2   0.999960   1.000040
1266              2   0.999930   1.000070
1272              2   0.999982   1.000018

[460 rows x 3 columns]


In [19]:
df_.to_pickle(f'/data/pgangwar/vaccinescout/data/pyro_ver/{ver}/PyroSpike.pkl')

In [21]:
count = (df_['R / R_A'] >= 1.025).sum()
print(f"Number of rows: {count}")

Number of rows: 105


In [28]:
a = df_[df_['R / R_A'] >= 1.025]
a.to_pickle('/data/pgangwar/vaccinescout/fit_mutations.pkl')